In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load the clean dataset we saved in Day 1
df = pd.read_csv('../data/cleaned_jobs.csv')

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nSample Key Skills:")
print(df['Key Skills'].head(3))

Dataset loaded successfully!
Shape: (28729, 11)

Sample Key Skills:
0                        media planning  digital media
1    pre sales  closing  software knowledge  client...
2    computer science  fabrication  quality check  ...
Name: Key Skills, dtype: object


In [2]:
# Fill any remaining nulls in Key Skills just to be safe
df['Key Skills'] = df['Key Skills'].fillna('')

# Create the TF-IDF Vectorizer
tfidf = TfidfVectorizer(stop_words='english')

# Fit and transform the Key Skills column
# This converts every job's skills into a row of numbers
tfidf_matrix = tfidf.fit_transform(df['Key Skills'])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)
print(f"\nThis means:")
print(f"  {tfidf_matrix.shape[0]} jobs")
print(f"  {tfidf_matrix.shape[1]} unique skills across all jobs")

TF-IDF Matrix Shape: (28729, 6521)

This means:
  28729 jobs
  6521 unique skills across all jobs


In [3]:
def recommend_jobs(user_skills, top_n=10):
    """
    Given a user's skills, return top N matching jobs.
    
    user_skills: string of skills e.g. "python machine learning sql"
    top_n: how many jobs to recommend
    """
    
    # Step 1: Clean the user input exactly like we cleaned job skills
    user_skills = user_skills.lower().strip()
    
    # Step 2: Convert user skills into a TF-IDF vector
    # Why? We need to express user skills in the same
    # "language" as our job matrix
    user_vector = tfidf.transform([user_skills])
    
    # Step 3: Calculate cosine similarity between
    # user vector and every job vector
    # This gives a similarity score between 0 and 1 for each job
    similarity_scores = cosine_similarity(user_vector, tfidf_matrix)
    
    # Step 4: Get indices of top N most similar jobs
    # argsort sorts from lowest to highest, so we reverse it
    top_indices = similarity_scores[0].argsort()[::-1][:top_n]
    
    # Step 5: Build results dataframe
    results = df.iloc[top_indices][['Job Title', 
                                    'Key Skills', 
                                    'Location', 
                                    'Industry',
                                    'Job Experience Required']].copy()
    
    results['Match Score'] = similarity_scores[0][top_indices]
    results['Match Score'] = results['Match Score'].apply(lambda x: f"{x*100:.1f}%")
    
    return results.reset_index(drop=True)

print("Recommendation function ready!")

Recommendation function ready!


In [4]:
# Test 1: Search for ML related jobs
print("=" * 60)
print("TEST 1: Searching for ML Engineer jobs")
print("=" * 60)
results1 = recommend_jobs("python machine learning data science numpy pandas")
print(results1.to_string())


TEST 1: Searching for ML Engineer jobs
                                                      Job Title                                                                                                                   Key Skills                  Location                                   Industry Job Experience Required Match Score
0                                       Data Science - Freshers                                                                                               data science  machine learning                 Hyderabad             IT-Software, Software Services               0 - 0 yrs       64.9%
1                                        Trainee-data Scientist                                                                                                     python  machine learning  Hyderabad,Pune,Delhi NCR             IT-Software, Software Services               0 - 5 yrs       59.1%
2   Immediate Opening for Python Developer @ Bangalore Location                

In [5]:
# Test 2: Search for web development jobs
print("=" * 60)
print("TEST 2: Searching for Web Developer jobs")
print("=" * 60)
results2 = recommend_jobs("javascript html css react nodejs")
print(results2[['Job Title', 'Location', 'Match Score']].to_string())

print()

# Test 3: Search for sales jobs
print("=" * 60)
print("TEST 3: Searching for Sales jobs")
print("=" * 60)
results3 = recommend_jobs("sales communication negotiation b2b")
print(results3[['Job Title', 'Location', 'Match Score']].to_string())

TEST 2: Searching for Web Developer jobs
                                                                Job Title           Location Match Score
0                                      Lead Software Engineer - Front End          Bengaluru       53.5%
1                     Front End JavaScript Developer - React.js/angularjs               Pune       53.1%
2                                                  UI Developer_ React.js               Pune       53.0%
3                                                    JavaScript Developer               Pune       52.0%
4   Job Opening for Tech Mahindra for Full Stack Engineers@pune 22nd June               Pune       52.0%
5                          Senior Web Engineer/developer - React.js/redux          Bengaluru       47.1%
6                      Frontend Engineer - Angular/ Javascript/ Html/ CSS             Mumbai       46.8%
7                             Need UI Developers For Singapore & Malaysia          Hyderabad       45.6%
8       Senior

In [8]:
import os
os.makedirs('../src', exist_ok=True)
print("src folder created!")

src folder created!


In [9]:
import pickle

# Save the TF-IDF vectorizer
with open('../src/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

# Save the TF-IDF matrix
with open('../src/tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(tfidf_matrix, f)

# Save the clean dataframe
df.to_csv('../data/cleaned_jobs.csv', index=False)

print("Model saved successfully!")


Model saved successfully!
